In [ ]:
# Libraries Installation 
%pip install requests pandas polars plotly sqlalchemy ipywidgets

In [1]:
# Cell 1: Imports and Environment Setup
import os
import logging
import requests
import pandas as pd
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
from sqlalchemy import create_engine, Column, String, Float, Integer
from sqlalchemy.orm import declarative_base, sessionmaker

# Ρύθμιση logging για καταγραφή των σταδίων εκτέλεσης
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("✅ All libraries have been uploaded")

✅ All libraries have been uploaded


In [ ]:
# Cell 2: Eurostat API Client Implementation

class EurostatClient:
    """Client για την άντληση στατιστικών δεδομένων NACE από το API της Eurostat."""
    
    BASE_URL = "https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data"

    def __init__(self, timeout: int = 30):
        self.timeout = timeout

    def fetch_dataset(self, dataset_code: str, params: dict = None) -> dict:
        """
        Πραγματοποιεί GET request στο API της Eurostat για ένα συγκεκριμένο dataset.
        
        :param dataset_code: Ο κωδικός του dataset (e.g., 'nama_10_a64')
        :param params: Προαιρετικές παράμετροι φιλτραρίσματος (format, geo, time κλπ)
        :return: JSON response ως λεξικό (dict)
        """
        # We ask for JSON-stat form
        default_params = {"format": "JSON", "lang": "en"}
        if params:
            default_params.update(params)

        url = f"{self.BASE_URL}/{dataset_code}"
        logger.info(f"Έναρξη λήψης δεδομένων από το URL: {url}")
        
        try:
            response = requests.get(url, params=default_params, timeout=self.timeout)
            response.raise_for_status()
            logger.info(f"✅ Downloaded Completed: {dataset_code}")
            return response.json()
        except requests.exceptions.RequestException as e:
            logger.error(f"❌ Connection Error of Eurostat's API: {e}")
            raise

# Test Client
client = EurostatClient()

In [ ]:
# Cell 3: Parsing & Dataset
import time
import itertools
import polars as pl

def parse_eurostat_json_to_polars(json_data):
    """Transform Eurostat JSON-stat format to Polars DataFrame."""
    dimensions = json_data.get('id', [])
    dim_categories = [
        list(json_data['dimension'][d]['category']['index'].keys()) 
        for d in dimensions
    ]
    cartesian_product = list(itertools.product(*dim_categories))
    values = json_data.get('value', {})
    
    records = []
    for i, combo in enumerate(cartesian_product):
        val = values.get(str(i), None)
        record = {dimensions[j]: combo[j] for j in range(len(dimensions))}
        record['value'] = val
        records.append(record)
        
    df = pl.DataFrame(records)
    # Clean and transform to Float64
    df_clean = (
        df.with_columns(pl.col('value').cast(pl.Float64))
        .drop_nulls(subset=['value'])
    )
    return df_clean


DATASET_CODE = "nama_10_a10"

# Countries List
target_countries = ["TR", "EL", "DE", "FR", "IT", "ES", "CY", "NL", "BE", "DK", "SE", "MD", "AT", "CH", "XK", "BA", "RS"]

all_dfs = []
print(f"🚀 Start download... {len(target_countries)}")

for country in target_countries:
    try:
        # Filter API request per country
        params = {
            "format": "JSON", 
            "lang": "en", 
            "geo": country,
            "unit": "CP_MEUR",   # Current prices, million euro
            "na_item": "B1G"     # Gross value added
        }
        raw_data = client.fetch_dataset(DATASET_CODE, params=params)
        
        # Transform to Polars DataFrame
        df_temp = parse_eurostat_json_to_polars(raw_data)
        all_dfs.append(df_temp)
        
        # 1 second pause to not overflow the API
        time.sleep(1)
        
    except Exception as e:
        print(f"⚠️ Error when downloaded {country}: {e}")

# Combine all dataframes
if all_dfs:
    df_nace = pl.concat(all_dfs)
    print(f"\n✅ Η Download Completed!")
    print(f"📊 The DataFrame has {len(df_nace)} recoerds.")
    print(f"🌍 Available Countries: {sorted(df_nace['geo'].unique().to_list())}")
    print(f"📅 Available Years: {sorted(df_nace['time'].unique().to_list())}")
else:
    print("\n❌ Fail. No data.")

2026-06-26 13:48:14,515 - INFO - Έναρξη λήψης δεδομένων από το URL: https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nama_10_a10
2026-06-26 13:48:14,687 - INFO - ✅ Downloaded Completed: nama_10_a10


🚀 Έναρξη τμηματικής λήψης για 17 περιοχές...


2026-06-26 13:48:15,703 - INFO - Έναρξη λήψης δεδομένων από το URL: https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nama_10_a10
2026-06-26 13:48:15,953 - INFO - ✅ Downloaded Completed: nama_10_a10
2026-06-26 13:48:16,957 - INFO - Έναρξη λήψης δεδομένων από το URL: https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nama_10_a10
2026-06-26 13:48:17,161 - INFO - ✅ Downloaded Completed: nama_10_a10
2026-06-26 13:48:18,164 - INFO - Έναρξη λήψης δεδομένων από το URL: https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nama_10_a10
2026-06-26 13:48:18,341 - INFO - ✅ Downloaded Completed: nama_10_a10
2026-06-26 13:48:19,345 - INFO - Έναρξη λήψης δεδομένων από το URL: https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/nama_10_a10
2026-06-26 13:48:19,592 - INFO - ✅ Downloaded Completed: nama_10_a10
2026-06-26 13:48:20,595 - INFO - Έναρξη λήψης δεδομένων από το URL: https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/d


✅ Η λήψη ολοκληρώθηκε επιτυχώς!
📊 Το συνολικό DataFrame περιέχει 6192 εγγραφές.
🌍 Διαθέσιμες Χώρες: ['AT', 'BA', 'BE', 'CH', 'CY', 'DE', 'DK', 'EL', 'ES', 'FR', 'IT', 'MD', 'NL', 'RS', 'SE', 'TR', 'XK']
📅 Διαθέσιμα Έτη: ['1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025']


In [ ]:
# Cell 4: SQLAlchemy Schema and Database Export
from sqlalchemy import create_engine, Column, Integer, String, Float
from sqlalchemy.orm import declarative_base

# Create Base class for ORM
Base = declarative_base()

class NaceStatistic(Base):
    __tablename__ = 'nace_statistics'
    
    id = Column(Integer, primary_key=True, autoincrement=True)
    nace_r2 = Column(String, index=True) # Index για γρήγορη αναζήτηση ανά κλάδο
    geo = Column(String, index=True)
    time = Column(String)
    unit = Column(String)
    value = Column(Float)

# Connect SQLite
DB_URI = 'sqlite:///eurostat_nace.db'
engine = create_engine(DB_URI)

Base.metadata.create_all(engine)

# Write data
try:
    df_nace.write_database(
        table_name="nace_statistics", 
        connection=DB_URI, 
        if_table_exists="replace"
    )
    print("✅ Data have been saved SQLite: eurostat_nace.db")
except Exception as e:
    print(f"❌ Error: {e}")

✅ Τα δεδομένα αποθηκεύτηκαν επιτυχώς στη βάση SQLite: eurostat_nace.db


In [6]:
# Cell 5: Final Custom HTML Dashboard (English UI, Billion € Formatting, Select All)
import pandas as pd
import json
from sqlalchemy import create_engine

# 1. Fetch Data
DB_URI = 'sqlite:///eurostat_nace.db'
engine = create_engine(DB_URI)
df = pd.read_sql("SELECT nace_r2, geo, time, value FROM nace_statistics ORDER BY time ASC", engine)

if df.empty:
    print("❌ No data found in the database!")
else:
    data_json = df.to_json(orient='records')
    
    countries = sorted(df['geo'].unique())
    nace_codes = sorted(df['nace_r2'].unique())
    years = sorted(df['time'].unique())

    # 2. Generate HTML Content (English localization & Value scaling)
    html_content = f"""
    <!DOCTYPE html>
    <html lang="en">
    <head>
        <meta charset="UTF-8">
        <title>Interactive NACE Dashboard</title>
        <script src="https://cdn.plot.ly/plotly-2.27.0.min.js"></script>
        <style>
            body {{ font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; margin: 20px; background-color: #f9f9f9; }}
            .dashboard-container {{ display: flex; gap: 25px; }}
            .sidebar {{ width: 300px; background: #ffffff; padding: 20px; border-radius: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }}
            .main-content {{ flex-grow: 1; background: #ffffff; padding: 20px; border-radius: 10px; box-shadow: 0 4px 6px rgba(0,0,0,0.1); }}
            h2 {{ color: #333; margin-top: 0; }}
            .filter-section {{ margin-bottom: 25px; }}
            .filter-section h3 {{ font-size: 15px; color: #555; border-bottom: 2px solid #eee; padding-bottom: 8px; margin-bottom: 8px; }}
            
            .btn-group {{ display: flex; gap: 5px; margin-bottom: 10px; }}
            .btn-group button {{ flex: 1; padding: 5px; font-size: 12px; cursor: pointer; border: 1px solid #ccc; background: #f1f1f1; border-radius: 4px; transition: background 0.2s; }}
            .btn-group button:hover {{ background: #e2e2e2; }}
            
            .checkbox-group {{ max-height: 220px; overflow-y: auto; display: flex; flex-direction: column; gap: 6px; padding-right: 5px; }}
            label {{ font-size: 14px; cursor: pointer; display: flex; align-items: center; gap: 8px; }}
            select {{ width: 100%; padding: 8px; border-radius: 5px; border: 1px solid #ccc; }}
        </style>
    </head>
    <body>
        <h2>NACE Economic Indicators Explorer</h2>
        <div class="dashboard-container">
            
            <div class="sidebar">
                <div class="filter-section">
                    <h3>🌍 Country / Region</h3>
                    <select id="countrySelect" onchange="renderChart()">
                        {"".join([f'<option value="{c}">{c}</option>' for c in countries])}
                    </select>
                </div>
                
                <div class="filter-section">
                    <h3>📊 NACE Sectors</h3>
                    <div class="btn-group">
                        <button onclick="toggleAll('nace-cb', true)">Select All</button>
                        <button onclick="toggleAll('nace-cb', false)">Clear</button>
                    </div>
                    <div class="checkbox-group" id="naceFilters">
                        {"".join([f'<label><input type="checkbox" class="nace-cb" value="{n}" checked onchange="renderChart()"> {n}</label>' for n in nace_codes])}
                    </div>
                </div>
                
                <div class="filter-section">
                    <h3>📅 Years</h3>
                    <div class="btn-group">
                        <button onclick="toggleAll('year-cb', true)">Select All</button>
                        <button onclick="toggleAll('year-cb', false)">Clear</button>
                    </div>
                    <div class="checkbox-group" id="yearFilters">
                        {"".join([f'<label><input type="checkbox" class="year-cb" value="{y}" checked onchange="renderChart()"> {y}</label>' for y in years])}
                    </div>
                </div>
            </div>
            
            <div class="main-content">
                <div id="plotlyChart" style="width: 100%; height: 600px;"></div>
            </div>
        </div>

        <script>
            const rawData = {data_json};

            function toggleAll(className, isChecked) {{
                const checkboxes = document.querySelectorAll('.' + className);
                checkboxes.forEach(cb => cb.checked = isChecked);
                renderChart();
            }}

            function renderChart() {{
                const selectedCountry = document.getElementById('countrySelect').value;
                const selectedNace = Array.from(document.querySelectorAll('.nace-cb:checked')).map(cb => cb.value);
                const selectedYears = Array.from(document.querySelectorAll('.year-cb:checked')).map(cb => String(cb.value));

                if (selectedNace.length === 0 || selectedYears.length === 0) {{
                    Plotly.react('plotlyChart', [], {{ title: 'Please select at least one sector and one year.' }});
                    return;
                }}

                const filteredData = rawData.filter(row => 
                    row.geo === selectedCountry && 
                    selectedNace.includes(row.nace_r2) && 
                    selectedYears.includes(String(row.time))
                );

                let traces = [];
                selectedNace.forEach(nace => {{
                    let naceData = filteredData.filter(row => row.nace_r2 === nace);
                    if(naceData.length > 0) {{
                        naceData.sort((a, b) => String(a.time).localeCompare(String(b.time)));
                        traces.push({{
                            x: naceData.map(row => row.time),
                            // Μετατροπή των Εκατομμυρίων σε Δισεκατομμύρια δια της διαίρεσης με το 1000
                            y: naceData.map(row => row.value / 1000),
                            name: nace,
                            mode: 'lines+markers',
                            type: 'scatter',
                            // Custom tooltip για να δείχνει "Billion €"
                            hovertemplate: '%{{y:.2f}} Billion €<extra>%{{name}}</extra>'
                        }});
                    }}
                }});

                const layout = {{
                    title: `Gross Value Added (GVA) for: ${{selectedCountry}}`,
                    xaxis: {{ title: 'Year', type: 'category' }},
                    yaxis: {{ title: 'Value (Billion €)' }}, // Ο άξονας πλέον το λέει ξεκάθαρα
                    hovermode: 'x unified',
                    template: 'plotly_white',
                    margin: {{ t: 50, l: 50, r: 20, b: 50 }}
                }};

                Plotly.react('plotlyChart', traces, layout);
            }}

            // Render immediately on load
            renderChart();
        </script>
    </body>
    </html>
    """

    # 3. Save the final file
    html_filename = "nace_dashboard_final.html"
    with open(html_filename, "w", encoding="utf-8") as f:
        f.write(html_content)
        
    print(f"🚀 SUCCESS! The final English dashboard is ready at: {html_filename}")

🚀 SUCCESS! The final English dashboard is ready at: nace_dashboard_final.html
